# Hamiltoanians and time evolution

MIMIQ provides tools to define quantum Hamiltonians and simulate their time evolution using Trotterization methods. This page explains how to construct Hamiltonians, compute their expectation values, and apply Lie-Trotter, Suzuki-Trotter, and Yoshida decompositions. 


### Hamiltonian
In quantum computing, Hamiltonians play a central role in algorithms such as the Variational Quantum Eigensolver (VQE), Quantum Phase Estimation, and the Quantum Approximate Optimization Algorithm (QAOA). They are used to encode the energy landscape of a physical system, generate dynamics, or define cost functions for optimization.

A typical Hamiltonian is expressed as a sum of weighted Pauli strings:
$$H = \sum_j c_j . P_j $$

Each term consists of a real coefficient $c_j$ and a Pauli string $P_j$, such as $X$, $ZZ$, or $XYZ$, which acts on a specific subset of qubits.

In quantum circuit frameworks, these Hamiltonians are often represented programmatically by associating each Pauli string with a set of target qubit indices and a coefficient, allowing users to construct and simulate physical models or optimization problems


In [1]:
from mimiqcircuits import *

### Building the Hamiltonian
#### example Ising Model
The 1D transverse-field Ising model is defined by the Hamiltonian:

$$H = -J\sum_{j=1}^{N-1}Z_j Z_{j+1} - h\sum_{j=1}^{N}X_j$$

This models a chain of spins with:

- nearest-neighbor interaction ($ZZ$ terms),
- and a transverse magnetic field ($X$ terms).

It’s widely used in condensed matter physics and quantum algorithm benchmarks.

In [2]:
N = 4
J = 1.0
h = 0.5

# creation of the hamiltonian
hamiltonian = Hamiltonian()

for j in range(N-1):
    _ = hamiltonian.push(-J, PauliString("ZZ"), j, j+1)

for j in range(N):
    _ = hamiltonian.push(-h, PauliString("X"), j)

print(hamiltonian)

4-qubit Hamiltonian with 7 terms:
├── -1.0 * ZZ @ q[0,1]
├── -1.0 * ZZ @ q[1,2]
├── -1.0 * ZZ @ q[2,3]
├── -0.5 * X @ q[0]
├── -0.5 * X @ q[1]
├── -0.5 * X @ q[2]
└── -0.5 * X @ q[3]


## Simulating time evolution

Suppose we want to apply $e^{-iHt} $ to a quantum state. This is useful for preparing ground states via imaginary-time evolution (approximate cooling), or evolving an initial state in real time.

Because H has non-commuting terms, we use a Trotter approximation.

Remind:
For $X$ and $Y$ two operators which don't commute. Trotter-Kato formula give the following expression:
$$e^{X+Y} = \lim_{n \to \infty } (e^{\frac{X}{n}}e^{\frac{Y}{n}})^n $$



In [6]:
# First order trotterization
c = Circuit()
c.push_lietrotter(hamiltonian, tuple(range(N)), t=0.1, steps=1)

4-qubit circuit with 1 instructions:
└── trotter @ q[0,1,2,3]

In [7]:
c.decompose()

4-qubit circuit with 7 instructions:
├── RZZ(-0.2) @ q[0,1]
├── RZZ(-0.2) @ q[1,2]
├── RZZ(-0.2) @ q[2,3]
├── RX(-0.1) @ q[0]
├── RX(-0.1) @ q[1]
├── RX(-0.1) @ q[2]
└── RX(-0.1) @ q[3]

In [8]:
c = Circuit()
c.push_suzukitrotter(hamiltonian, tuple(range(N)), t=0.1, steps=1, order=2)

c.decompose()

4-qubit circuit with 14 instructions:
├── RZZ(-0.1) @ q[0,1]
├── RZZ(-0.1) @ q[1,2]
├── RZZ(-0.1) @ q[2,3]
├── RX(-0.05) @ q[0]
├── RX(-0.05) @ q[1]
├── RX(-0.05) @ q[2]
├── RX(-0.05) @ q[3]
├── RX(-0.05) @ q[3]
├── RX(-0.05) @ q[2]
├── RX(-0.05) @ q[1]
├── RX(-0.05) @ q[0]
├── RZZ(-0.1) @ q[2,3]
├── RZZ(-0.1) @ q[1,2]
└── RZZ(-0.1) @ q[0,1]

## Measuring the energy

Once the circuit has prepared the desired quantum state such as by applying time evolution with Trotterization we can measure the energy by evaluating the expectation value (push_expval()) of the Hamiltonian.

In [9]:
c.push_expval(hamiltonian, *range(N))

4-qubit, 7-zvar circuit with 16 instructions:
├── suzukitrotter_2 @ q[0,1,2,3]
├── ⟨ZZ⟩ @ q[0,1], z[0]
├── z[0] *= -1.0
├── ⟨ZZ⟩ @ q[1,2], z[1]
├── z[1] *= -1.0
├── ⟨ZZ⟩ @ q[2,3], z[2]
├── z[2] *= -1.0
├── ⟨X⟩ @ q[0], z[3]
├── z[3] *= -0.5
├── ⟨X⟩ @ q[1], z[4]
├── z[4] *= -0.5
├── ⟨X⟩ @ q[2], z[5]
├── z[5] *= -0.5
├── ⟨X⟩ @ q[3], z[6]
├── z[6] *= -0.5
└── z[0] += 0.0 + z[1] + z[2] + z[3] + z[4] + z[5] + z[6]